# Modelo de Machine Learning - Detección de Fraude en Siniestros
Este notebook tiene como objetivo entrenar un modelo híbrido para la detección de fraude basado en los datos sintéticos generados. Utilizaremos un Random Forest Classifier utilizando la `etiqueta_fraude_simulada` como target.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import os


## 1. Carga de Datos

In [2]:
# Cargar datos desde los CSV generados
df_sin = pd.read_csv('../data/synthetic/siniestros.csv')
df_pol = pd.read_csv('../data/synthetic/polizas.csv')
df_ase = pd.read_csv('../data/synthetic/asegurados.csv')

# Unir datos relevantes (ej. score_cliente de asegurado)
df_merged = df_sin.merge(df_pol[['id_poliza', 'prima', 'suma_asegurada']], on='id_poliza', how='left')
df_merged = df_merged.merge(df_ase[['id_asegurado', 'score_cliente']], on='id_asegurado', how='left')

print(f"Total de siniestros: {len(df_merged)}")
print(f"Fraudes simulados: {df_merged['etiqueta_fraude_simulada'].sum()}")


Total de siniestros: 1000
Fraudes simulados: 208


## 2. Ingeniería de Características (Feature Engineering)

In [3]:
# Características (Features) a utilizar en el modelo
features = [
    'monto_reclamado', 
    'dias_desde_inicio_poliza', 
    'dias_desde_fin_poliza', 
    'dias_entre_ocurrencia_reporte', 
    'historial_siniestros_asegurado',
    'score_cliente',
    'documentos_completos' # boolean
]

X = df_merged[features].copy()
X['documentos_completos'] = X['documentos_completos'].astype(int)

y = df_merged['etiqueta_fraude_simulada']

X.fillna(0, inplace=True) # Rellenar nulos si existieran


,monto_reclamado,dias_desde_inicio_poliza,dias_desde_fin_poliza,dias_entre_ocurrencia_reporte,historial_siniestros_asegurado,score_cliente,documentos_completos
0,37033.70,302,63,0,2,44,1
1,2903.90,52,313,1,2,58,1
2,5992.75,247,118,1,0,99,1
3,5371.69,33,332,0,0,96,1
4,5111.44,329,36,1,1,37,1
...,...,...,...,...,...,...,...
995,4628.41,283,82,1,0,95,1
996,3766.78,38,327,1,0,56,0
997,2472.60,317,48,0,2,40,1
998,5646.23,189,176,1,0,34,1


## 3. Entrenamiento del Modelo (Random Forest)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Confusion Matrix:
[[157   1]
 [ 13  29]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.99      0.96       158
           1       0.97      0.69      0.81        42

    accuracy                           0.93       200
   macro avg       0.95      0.84      0.88       200
weighted avg       0.93      0.93      0.93       200



## 4. Exportar el Modelo
Guardaremos el modelo para ser utilizado en el Backend (`app/model/fraud_model.py`).

In [5]:
os.makedirs('../app/model', exist_ok=True)
joblib.dump(rf_model, '../app/model/random_forest_fraud.joblib')
print("Modelo guardado en 'app/model/random_forest_fraud.joblib'")

# Exportar también la lista de features para asegurar consistencia en la inferencia
joblib.dump(features, '../app/model/model_features.joblib')


Modelo guardado en 'app/model/random_forest_fraud.joblib'


['../app/model/model_features.joblib']